# 🧪 Agent Server 实时测试

测试**真实 agent server**（非离线），通过 HTTP 发送 RunAgentInput，验证工具调用链路。

| 模式 | 端口 | 说明 |
|------|------|------|
| 直连 | `:8000` | 跳过 APISIX，需手动传 `X-Org-Name` |
| APISIX | `:9080` | 走完整生产路径，APISIX 注入 `X-Org-Name` |

**前置**：`python3 -m uvicorn main:app --host 0.0.0.0 --port 8000 >> logs/agent.log 2>&1 &`

## 0. 初始化

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from nb_utils import setup_env, get_token, make_payload, run_agent, print_events
setup_env()

import config
print('GATEWAY_URL:', config.GATEWAY_URL)

AGENT_DIRECT  = 'http://127.0.0.1:8000/copilotkit/admin'
AGENT_APISIX  = f'{config.GATEWAY_URL}/copilotkit/admin'
print('直连  :', AGENT_DIRECT)
print('APISIX:', AGENT_APISIX)

GATEWAY_URL: http://localhost:9080
直连  : http://127.0.0.1:8000/copilotkit/admin
APISIX: http://localhost:9080/copilotkit/admin


## 1. 登录获取 Token

In [2]:
# ── 填入你的账号 ──────────────────────────────────────────
USERNAME = 'luke'
PASSWORD = 'jmx931228'
# ─────────────────────────────────────────────────────────

TOKEN, ORG = await get_token(USERNAME, PASSWORD)
print('ORG  :', ORG)
print('TOKEN:', TOKEN[:60], '...')

✅ 登录成功  org=lukeco  token 有效期=21600s
ORG  : lukeco
TOKEN: Bearer eyJhbGciOiJFUzI1NiIsInR5cCI6IkpXVCJ9.eyJ1c2VyX2lkIjoi ...


## 2. 测试：直连 agent :8000

In [3]:
MSG = '列出我所有的项目'   # ← 修改这里

payload = make_payload(MSG, org=ORG, layer='org')
events  = await run_agent(AGENT_DIRECT, TOKEN, payload, org=ORG)  # 直连需要 X-Org-Name
print_events(events)

共 570 个 SSE 事件

🚀 RUN_STARTED
▶  STEP  agent
▶  STEP  tools
  🔧 TOOL_CALL_START  name=list_projects
  ✅ TOOL_CALL_END
  📦 RESULT  [{"id": "onboardceshi", "slug": "onboardceshi", "title": "onboard测试", "description": "", "status": "ACTIVE", "createdAt"
▶  STEP  agent
▶  STEP  tools
▶  STEP  agent
🏁 RUN_FINISHED

── AI 回复 ─────────────────────────────────────────
好的，我来查询您组织下的所有项目。您当前组织 **lukeco** 下共有以下项目：

| 项目名称 | 标识符 | 状态 |
|---------|--------|------|
| 📁 **onboard测试** | `onboardceshi` | ✅ 活跃 |
| 📁 **abcde** | `abcde` | ✅ 活跃 |
| 📁 **pro1** | `pro1` | 📦 已归档 |

您可以点击下方按钮快速跳转到对应项目：好的，以上是您组织下的所有项目列表。请问您想进入哪个项目查看详情？
────────────────────────────────────────────────────


## 3. 测试：走 APISIX :9080（完整生产路径）

In [4]:
MSG = '列出我所有的项目'   # ← 修改这里

payload = make_payload(MSG, org=ORG, layer='org')
events  = await run_agent(AGENT_APISIX, TOKEN, payload)  # APISIX 自动注入 X-Org-Name
print_events(events)

共 838 个 SSE 事件

🚀 RUN_STARTED
▶  STEP  agent
▶  STEP  tools
  🔧 TOOL_CALL_START  name=list_projects
  ✅ TOOL_CALL_END
  📦 RESULT  [{"id": "onboardceshi", "slug": "onboardceshi", "title": "onboard测试", "description": "", "status": "ACTIVE", "createdAt"
▶  STEP  agent
▶  STEP  tools
▶  STEP  agent
🏁 RUN_FINISHED

── AI 回复 ─────────────────────────────────────────
好的，我来查询您组织下的所有项目。您当前组织 **lukeco** 下共有以下项目：

| 项目名称 | 标识符 | 状态 |
|---------|--------|:----:|
| 📁 **onboard测试** | `onboardceshi` | ✅ 活跃 |
| 📁 **abcde** | `abcde` | ✅ 活跃 |
| 📁 **pro1** | `pro1` | 🗂️ 已归档 |

您可以点击下方按钮快速跳转到对应项目：抱歉，当前环境暂不支持导航按钮功能。以下是项目列表供您参考：

---

### 📋 组织「lukeco」下的项目

| # | 项目名称 | 标识符 (Slug) | 状态 |
|:-:|---------|:-------------:|:----:|
| 1 | **onboard测试** | `onboardceshi` | ✅ 活跃 |
| 2 | **abcde** | `abcde` | ✅ 活跃 |
| 3 | **pro1** | `pro1` | 🗂️ 已归档 |

> 💡 **提示**：您可以直接告诉我想进入哪个项目，例如"进入 onboard测试"，我可以帮您进一步查看该项目下的数据库和模型。
────────────────────────────────────────────────────


## 4. 测试：Project 层（需要 project_slug）

In [5]:
PROJECT_SLUG = 'abcde'          # ← 填入要测试的项目 slug
MSG          = '列出所有数据库'   # ← 修改这里

payload = make_payload(MSG, org=ORG, layer='project', project_slug=PROJECT_SLUG)
events  = await run_agent(AGENT_APISIX, TOKEN, payload)
print_events(events)

共 503 个 SSE 事件

🚀 RUN_STARTED
▶  STEP  agent
▶  STEP  tools
  🔧 TOOL_CALL_START  name=list_databases
  ✅ TOOL_CALL_END
  📦 RESULT  ["casdoor", "mc_meta_dev", "mc_private_myproject", "modelcraft", "modelcraft_dev", "modelcraft_strapi", "modelcraft_test
▶  STEP  agent
🏁 RUN_FINISHED

── AI 回复 ─────────────────────────────────────────
好的，我来查看当前项目（abcde）中有哪些数据库。当前项目 **abcde** 中共有以下数据库：

| 序号 | 数据库名称 |
|:---:|:---|
| 1 | `casdoor` |
| 2 | `mc_meta_dev` |
| 3 | `mc_private_myproject` |
| 4 | `modelcraft` |
| 5 | `modelcraft_dev` |
| 6 | `modelcraft_strapi` |
| 7 | `modelcraft_test_dev` |
| 8 | `private_myproject` |
| 9 | `test_db` |
| 10 | `test_schema_validation_1771158828` |
| 11 | `test_schema_validation_1771158835` |
| 12 | `test_schema_validation_1771158841` |
| 13 | `test_user_setup` |

共 **13** 个数据库。如果您想查看某个数据库中的模型（数据表），请告诉我数据库名称，我来进一步查询。
────────────────────────────────────────────────────


## 5. 批量测试多条消息

In [6]:
from nb_utils import make_payload, run_agent

CASES = [
    ('列出我所有的项目',                       'org',     ''),
    ('帮我跳转到 abcde 项目',                  'org',     ''),
    ('abcde 项目里有哪些数据库',               'project', 'abcde'),
]

for msg, layer, project in CASES:
    payload = make_payload(msg, org=ORG, layer=layer, project_slug=project)
    events  = await run_agent(AGENT_APISIX, TOKEN, payload)

    # 统计工具调用
    tools = [e.get('toolCallName','') for e in events if 'TOOL_CALL_START' in e.get('type','')]
    text  = ''.join(e.get('delta','') for e in events if 'TEXT_MESSAGE_CONTENT' in e.get('type',''))

    print(f'Q: {msg!r}')
    print(f'   tools : {tools or ["(none)"]}')
    print(f'   reply : {text[:100]}')
    print()

Q: '列出我所有的项目'
   tools : ['list_projects']
   reply : 好的，我来查询您组织下的所有项目。您当前组织 **lukeco** 下共有以下项目：

| 项目名称 | 标识符 | 状态 |
|---------|--------|------|
| 📁 **on

Q: '帮我跳转到 abcde 项目'
   tools : ['list_projects']
   reply : 好的，我来查找项目信息并为您导航。找到了 **abcde** 项目，为您跳转过去。已找到 **abcde** 项目，您可以直接点击下方链接跳转：

🔗 [点击进入 abcde 项目 →](/org/l

Q: 'abcde 项目里有哪些数据库'
   tools : ['list_databases']
   reply : 好的，我来查看当前项目 `abcde` 中有哪些数据库。项目 **abcde** 中共有以下数据库：

| 序号 | 数据库名称 |
|:---:|:---|
| 1 | `casdoor` |
| 



## 7. ui_present_proposal 专项测试

测试三类意图：

| # | 消息 | 期望行为 |
|---|------|----------|
| A | 当前有哪些项目 | `list_projects` → 每个项目一个 `action_candidate` |
| B | 帮我去数据模型管理 | 直接 `ui_present_proposal`，无多余工具调用 |
| C | 帮我配置权限 | 直接 `ui_present_proposal`，**不调** `list_databases` |

In [7]:
from nb_utils import make_nav_payload, print_nav_proposal, print_events, NAV_TOOLS

# ── 当前页面 aiTargets（Org workspace 页） ─────────────────────────────────────
AI_TARGETS_ORG = [
    {"id": "create_project", "label": "新建项目", "description": "点击打开新建项目表单"},
]

PROJECT_SLUG = 'onboardceshi'    # ← 按需修改

print('NAV_TOOLS 已加载:', [t['name'] for t in NAV_TOOLS])
print('PROJECT_SLUG    :', PROJECT_SLUG)

NAV_TOOLS 已加载: ['show_toast', 'ui_present_proposal']
PROJECT_SLUG    : onboardceshi


### 7-A  当前有哪些项目（期望：每个项目 → action_candidate）

In [8]:
payload_a = make_nav_payload(
    '当前有哪些项目',
    org=ORG, layer='org',
    ai_targets=AI_TARGETS_ORG,
)
events_a = await run_agent(AGENT_APISIX, TOKEN, payload_a)

print_events(events_a)
print()
print_nav_proposal(events_a)

共 514 个 SSE 事件

🚀 RUN_STARTED
▶  STEP  agent
▶  STEP  tools
  🔧 TOOL_CALL_START  name=list_projects
  ✅ TOOL_CALL_END
  📦 RESULT  [{"id": "onboardceshi", "slug": "onboardceshi", "title": "onboard测试", "description": "", "status": "ACTIVE", "createdAt"
▶  STEP  agent
🏁 RUN_FINISHED

── AI 回复 ─────────────────────────────────────────
好的，我来查询当前组织下的项目列表。当前组织 **lukeco** 下共有以下项目：

| 项目名称 | 标识（Slug） | 状态 |
|---------|-------------|------|
| 📁 **onboard测试** | `onboardceshi` | ✅ 活跃 |
| 📁 **abcde** | `abcde` | ✅ 活跃 |
| 📁 **pro1** | `pro1` | 📦 已归档 |

点击下方按钮可快速跳转到对应项目：
────────────────────────────────────────────────────

  ⚠️  未找到 ui_present_proposal 调用


In [ ]:
import json
from pathlib import Path

# 组装成一个完整 JSON 对象（含请求与事件列表）
event_stream_json = {
    "request": payload_a["messages"][0]["content"],
    "threadId": payload_a.get("threadId"),
    "runId": payload_a.get("runId"),
    "eventCount": len(events_a),
    "events": events_a,
}

# Notebook 里直接看
print(json.dumps(event_stream_json, ensure_ascii=False, indent=2))

# 可选：落盘，方便你用编辑器查看
out = Path("event_stream_a.json")
out.write_text(json.dumps(event_stream_json, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"已写入: {out.resolve()}")

preview = [
    {
        "type": e.get("type"),
        "stepName": e.get("stepName"),
        "toolCallName": e.get("toolCallName"),
        "delta_preview": (e.get("delta", "")[:80] if isinstance(e.get("delta"), str) else None),
    }
    for e in events_a
]
print(json.dumps(preview, ensure_ascii=False, indent=2))

IndentationError: unexpected indent (1682519714.py, line 2)

### 7-B  帮我去数据模型管理（期望：直接导航，无额外工具调用）

In [9]:
payload_b = make_nav_payload(
    '帮我去数据模型管理',
    org=ORG, layer='org', project_slug=PROJECT_SLUG,
    ai_targets=AI_TARGETS_ORG,
)
events_b = await run_agent(AGENT_APISIX, TOKEN, payload_b)

print_events(events_b)
print()
print_nav_proposal(events_b)

共 160 个 SSE 事件

🚀 RUN_STARTED
▶  STEP  agent
🏁 RUN_FINISHED

── AI 回复 ─────────────────────────────────────────
好的，我来帮您导航到数据模型编辑器页面。
────────────────────────────────────────────────────

  ⚠️  未找到 ui_present_proposal 调用


### 7-C  帮我配置权限（期望：直接导航，不调 list_databases）

In [10]:
payload_c = make_nav_payload(
    '帮我配置权限',
    org=ORG, layer='org', project_slug=PROJECT_SLUG,
    ai_targets=AI_TARGETS_ORG,
)
events_c = await run_agent(AGENT_APISIX, TOKEN, payload_c)

print_events(events_c)
print()
print_nav_proposal(events_c)

共 162 个 SSE 事件

🚀 RUN_STARTED
▶  STEP  agent
🏁 RUN_FINISHED

── AI 回复 ─────────────────────────────────────────
好的，我来帮您配置权限。请问您具体想配置哪方面的权限呢？

1. **RBAC 角色管理** - 管理角色及其权限
2. **RBAC 用户授权** - 为用户分配角色
3. **权限包管理** - 管理权限包

请告诉我您想进入哪个页面？
────────────────────────────────────────────────────

  ⚠️  未找到 ui_present_proposal 调用


### 7-D  自定义消息（随时改这里手动试）

In [11]:
# ← 改这里测任意消息
MSG          = '帮我新建一个项目'
LAYER        = 'org'            # 'org' 或 'project'
PROJECT_SLUG = 'onboardceshi'   # layer='project' 时有效

payload_d = make_nav_payload(
    MSG, org=ORG, layer=LAYER, project_slug=PROJECT_SLUG,
    ai_targets=AI_TARGETS_ORG,
)
events_d = await run_agent(AGENT_APISIX, TOKEN, payload_d)

print_events(events_d)
print()
print_nav_proposal(events_d)

共 215 个 SSE 事件

🚀 RUN_STARTED
▶  STEP  agent
🏁 RUN_FINISHED

── AI 回复 ─────────────────────────────────────────
好的，我来帮您新建项目。不过新建项目需要在界面上手动操作，我来引导您到新建项目的位置。
────────────────────────────────────────────────────

  ⚠️  未找到 ui_present_proposal 调用


## 6. 查看最新日志（可选）

In [12]:
import subprocess
LOG = os.path.join(os.path.abspath('..'), 'logs', 'agent.log')
result = subprocess.run(['tail', '-30', LOG], capture_output=True, text=True)

import json
for line in result.stdout.strip().split('\n'):
    try:
        d = json.loads(line)
        ev = d.get('event', '')
        if ev in ('tool.call.start', 'tool.call.end', 'tool.error', 'agent.output', 'agent.event'):
            ts = d.get('timestamp','')[-8:]  # HH:MM:SS
            if ev == 'tool.call.start':
                print(f'[{ts}] 🔧 {ev:20s}  {d.get("tool_name","")}  args={d.get("args_summary","")}')
            elif ev == 'tool.call.end':
                ok = '✅' if d.get('success') else '❌'
                print(f'[{ts}] {ok} {ev:20s}  {d.get("tool_name","")}  {d.get("duration_ms",0)}ms')
            elif ev == 'tool.error':
                print(f'[{ts}] ❌ {ev:20s}  {d.get("tool_name","")}  {d.get("error_type","")}:{d.get("error_msg","")[:80]}')
            elif ev == 'agent.output':
                print(f'[{ts}] 💬 {ev:20s}  {d.get("content_preview","")[:80]}')
            elif ev == 'agent.event':
                print(f'[{ts}] 📌 {ev:20s}  {d.get("event_type","")}')
    except:
        pass

[.664346Z] 📌 agent.event           RUN_STARTED
[.666888Z] 📌 agent.event           STEP_STARTED
[.619043Z] 📌 agent.event           STATE_SNAPSHOT
[.620930Z] 📌 agent.event           STEP_FINISHED
[.621225Z] 📌 agent.event           STATE_SNAPSHOT
[.621661Z] 📌 agent.event           RUN_FINISHED
[.661225Z] 📌 agent.event           RUN_STARTED
[.663463Z] 📌 agent.event           STEP_STARTED
[.973793Z] 📌 agent.event           STATE_SNAPSHOT
[.975616Z] 📌 agent.event           STEP_FINISHED
[.975904Z] 📌 agent.event           STATE_SNAPSHOT
[.976331Z] 📌 agent.event           RUN_FINISHED
[.016513Z] 📌 agent.event           RUN_STARTED
[.019055Z] 📌 agent.event           STEP_STARTED
[.280892Z] 📌 agent.event           STATE_SNAPSHOT
[.282863Z] 📌 agent.event           STEP_FINISHED
[.283162Z] 📌 agent.event           STATE_SNAPSHOT
[.283625Z] 📌 agent.event           RUN_FINISHED
